# Crop2ML composition generation and validation

This notebook validates the Monica composition algorithm, resolves its ModelUnit interfaces, generates the corresponding Crop2ML composition XML, and verifies the generated links.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import sys
import xml.etree.ElementTree as ET
import pycropml
from pycropml.transpiler.antlr_py.composition_compiler import (
    compile_composition,
    find_composition_algorithm,
    write_composition_xml,
)
from pycropml.transpiler.antlr_py.validate_composition import validate

PROJECT = Path(pycropml.__file__).resolve().parents[2]

CROP2ML = PROJECT / "example" / "Monica_SoilTemp" / "crop2ml"
ALGORITHM = find_composition_algorithm(CROP2ML)

assert ALGORITHM.is_file(), ALGORITHM
ALGORITHM

PosixPath('/mnt/d/Docs/PyCropML_Old/example/Monica_SoilTemp/crop2ml/algo/pyx/SoilTemperatureComp.pyx')

## Validate syntax, YAML metadata, and ModelUnit references

In [2]:
assert validate(ALGORITHM, CROP2ML)
print("Composition algorithm is valid.")

/mnt/d/Docs/PyCropML_Old/example/Monica_SoilTemp/crop2ml/algo/pyx/SoilTemperatureComp.pyx: valid
Composition algorithm is valid.


## Compile the algorithm to models and links

In [3]:
composition, model_directory = compile_composition(ALGORITHM, CROP2ML)

assert composition.metadata["name"] == "SoilTemperatureComp"
assert composition.models == [
    "NoSnowSoilSurfaceTemperature",
    "WithSnowSoilSurfaceTemperature",
    "SoilTemperature",
]
assert len(composition.internal_links) == 2
assert len(composition.output_links) == 2

print("Models:", composition.models)
print("Input links:", len(composition.input_links))
print("Internal links:", composition.internal_links)
print("Output links:", composition.output_links)

Models: ['NoSnowSoilSurfaceTemperature', 'WithSnowSoilSurfaceTemperature', 'SoilTemperature']
Input links: 42
Internal links: [{'source': 'NoSnowSoilSurfaceTemperature.soilSurfaceTemperature', 'target': 'WithSnowSoilSurfaceTemperature.noSnowSoilSurfaceTemperature'}, {'source': 'WithSnowSoilSurfaceTemperature.soilSurfaceTemperature', 'target': 'SoilTemperature.soilSurfaceTemperature'}]
Output links: [{'source': 'WithSnowSoilSurfaceTemperature.soilSurfaceTemperature', 'target': 'soilSurfaceTemperature'}, {'source': 'SoilTemperature.soilTemperature', 'target': 'soilTemperature'}]


## Generate and validate the composition XML

The XML is written to a temporary directory so running the notebook does not overwrite the reference composition.

In [4]:
with TemporaryDirectory() as temporary_directory:
    output = Path(temporary_directory) / "composition.SoilTemperatureComp.xml"
    generated = write_composition_xml(
        ALGORITHM,
        output_file=output,
        crop2ml_directory=CROP2ML,
    )

    root = ET.parse(generated).getroot()
    models = root.findall("./Composition/Model")
    input_links = root.findall("./Composition/Links/InputLink")
    internal_links = root.findall("./Composition/Links/InternalLink")
    output_links = root.findall("./Composition/Links/OutputLink")

    assert root.tag == "ModelComposition"
    assert root.attrib["name"] == "SoilTemperatureComp"
    assert root.attrib["id"] == "Monica_SoilTemp.SoilTemperatureComp"
    assert len(models) == 3
    assert len(input_links) == len(composition.input_links)
    assert len(internal_links) == 2
    assert len(output_links) == 2

    generated_xml = generated.read_text(encoding="utf-8")

print("Generated XML passed all assertions.")
print(generated_xml)
print(len(input_links), len(internal_links), len(output_links))

Generated XML passed all assertions.
<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE ModelComposition PUBLIC " " "https://raw.githubusercontent.com/AgriculturalModelExchangeInitiative/crop2ml/master/ModelComposition.dtd">
<ModelComposition name="SoilTemperatureComp" id="Monica_SoilTemp.SoilTemperatureComp" version="1" timestep="1">
    <Description>
        <Title>SoilTemperature model</Title>
        <Authors>Michael Berg-Mohnicke</Authors>
        <Institution>ZALF e.V.</Institution>
        <Reference />
        <ExtendedDescription />
        <ShortDescription>Calculates the soil temperature in all layers and soil surface temperature.
</ShortDescription>
    </Description>
    <Algorithm language="cyml-composition" filename="algo/pyx/SoilTemperatureComp.pyx" />
    <Composition>
        <Model name="NoSnowSoilSurfaceTemperature" id="Monica_SoilTemp.NoSnowSoilSurfaceTemperature" filename="unit.NoSnowSoilSurfaceTemperature.xml" />
        <Model name="WithSnowSoilSurfaceTemperature"